In [0]:
# Databricks notebook source
# ETL - Squad 3 - ecommerce_clientes
# Fluxo: Raw CSV -> Bronze Delta -> Silver Delta -> Gold Delta/SQL Server

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
SOURCE_FILE = "ecommerce_clientes.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false",
}

EXPECTED_COLUMNS = [
    "id_cliente",
    "uuid_cliente",
    "nome",
    "sobrenome",
    "email",
    "senha_hash",
    "dt_cadastro",
    "dt_ultima_atualizacao",
]

KEY_COLUMNS = ["id_cliente"]

BRONZE_TABLE = "ecommerce_clientes"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

PARTITION_DATE_COLUMN = "dt_cadastro"
BRONZE_WRITE_MODE = "overwrite"

print("SOURCE_PATH:", SOURCE_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("PARTITION_DATE_COLUMN:", PARTITION_DATE_COLUMN)

In [0]:
adls_options = get_adls_options()

print("Opções ADLS configuradas.")

In [0]:
# ler CSV da Raw

df_source = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

# validar colunas esperadas
actual_columns = df_source.columns

missing_columns = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_columns = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

if missing_columns:
    raise Exception(f"Colunas obrigatórias ausentes na origem: {missing_columns}")

if extra_columns:
    print(f"Atenção: existem colunas extras na origem: {extra_columns}")
else:
    print("Validação OK: todas as colunas esperadas foram encontradas.")

df_source.printSchema()
display(df_source.limit(10))

In [0]:
# contar origem

total_source = df_source.count()

print(f"Total de registros lidos da Raw: {total_source}")

In [0]:
# validar conversão da data de particionamento

from pyspark.sql.functions import col, to_timestamp, count, when

df_test_date = df_source.withColumn(
    "dt_cadastro_convertida",
    to_timestamp(col(PARTITION_DATE_COLUMN))
)

df_validacao_data = df_test_date.select(
    count("*").alias("total_linhas"),
    count(when(col(PARTITION_DATE_COLUMN).isNull(), True)).alias("dt_cadastro_nula_origem"),
    count(
        when(
            col(PARTITION_DATE_COLUMN).isNotNull() &
            col("dt_cadastro_convertida").isNull(),
            True
        )
    ).alias("falhas_conversao")
)

display(df_validacao_data)

validacao_data = df_validacao_data.collect()[0]

if validacao_data["falhas_conversao"] > 0:
    raise Exception("Existem valores de dt_cadastro que não foram convertidos para timestamp.")

print("Validação OK: dt_cadastro pode ser usada para particionamento.")

In [0]:
# criar DataFrame Bronze

from pyspark.sql.functions import current_timestamp, year, month

df_bronze = (
    df_source
    .withColumn("bronze_source_file", col("_metadata.file_path"))
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("_partition_date", to_timestamp(col(PARTITION_DATE_COLUMN)))
    .withColumn("ano", year(col("_partition_date")))
    .withColumn("mes", month(col("_partition_date")))
    .drop("_partition_date")
)

In [0]:
# visualizar Bronze antes de gravar

display(
    df_bronze
    .select(
        "id_cliente",
        "dt_cadastro",
        "ano",
        "mes",
        "bronze_ingested_at",
        "bronze_source_file"
    )
    .limit(20)
)

In [0]:
# validar particionamento Bronze
display(
    df_bronze.select(
        count("*").alias("total_linhas"),
        count(when(col("ano").isNull(), True)).alias("ano_nulo"),
        count(when(col("mes").isNull(), True)).alias("mes_nulo")
    )
)

In [0]:
# gravar Bronze Delta

(
    df_bronze
    .write
    .format("delta")
    .options(**adls_options)
    .mode(BRONZE_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(BRONZE_PATH)
)

print(f"Dados gravados com sucesso na Bronze: {BRONZE_PATH}")

In [0]:
# ler Bronze gravada

df_bronze_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

df_bronze_saved.printSchema()

total_bronze = df_bronze_saved.count()

print(f"Total de registros na Bronze: {total_bronze}")

display(df_bronze_saved.limit(10))

In [0]:
# validar origem x Bronze

print(f"Total origem Raw: {total_source}")
print(f"Total gravado Bronze: {total_bronze}")

if total_source != total_bronze:
    raise Exception("Erro: quantidade de registros diferente entre Raw e Bronze.")

print("Validação OK: quantidade de registros da Raw e da Bronze é igual.")

# validar partições após gravação
df_validacao_bronze = df_bronze_saved.select(
    count("*").alias("total_linhas"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo"),
    count(when(col("bronze_ingested_at").isNull(), True)).alias("bronze_ingested_at_nulo"),
    count(when(col("bronze_source_file").isNull(), True)).alias("bronze_source_file_nulo")
)

display(df_validacao_bronze)

validacao_bronze = df_validacao_bronze.collect()[0]

if validacao_bronze["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo na Bronze.")

if validacao_bronze["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo na Bronze.")

if validacao_bronze["bronze_ingested_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_ingested_at.")

if validacao_bronze["bronze_source_file_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_source_file.")

print("Validação final da Bronze OK.")